# Configurações & Preparação

In [0]:
titanic_df = (
    spark.read.csv(
        "/Volumes/workspace/default/data/raw/titanic/train.csv",
        header=True, 
        inferSchema=True
    )
)

display(titanic_df)

# Preparação de Dados

## VectorAssembler

- Todos os algoritmos de ML do Spark esperam que os dados de entrada estejam em **uma única coluna do tipo** `Vector`

O VectorAssembler aceita três tipos:

- Numérico (`IntegerType`, `DoubleType`, `FloatType`, etc.) — cada coluna vira um elemento do vetor.
- Booleano (`BooleanType`) — convertido para 1.0 (true) ou 0.0 (false).
- Vetor (`VectorUDT`) — os elementos do vetor são "desempacotados" e adicionados inline. Isso é útil quando você já fez alguma transformação anterior (como um OneHotEncoder) que retorna um vetor.

> **IMPORTANTE:** Por padrão, o VectorAssembler lança uma exceção se encontrar nulos. 
> Você controla esse comportamento com o parâmetro handleInvalid
> - `error`: lança exceção ao encontrar nulo
> - `skip`: remove a linha inteira que contém nulo
> - `keep`: substitui o nulo por `NaN` no vetor

## StringIndexer

É um transformador do Spark MLlib que converte uma coluna de strings em uma coluna de índices numéricos. Os algoritmos de ML não conseguem trabalhar com texto diretamente — o StringIndexer resolve isso transformando categorias em números.

> A atribuição dos índices é feita por frequência: a categoria mais frequente recebe o índice 0.0, a segunda mais frequente recebe 1.0, e assim por diante.

In [0]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder

# Indexa a coluna 'Sex' (É possível realizar para várias coluans de uma vez)
sex_indexer = StringIndexer(inputCol='Sex', outputCol='Sex_index', handleInvalid='keep')
titanic_df_indexed = sex_indexer.fit(titanic_df).transform(titanic_df)

# Aplica one-hot encoding
sex_encoder = OneHotEncoder(inputCol='Sex_index', outputCol='Sex_onehot', handleInvalid='keep')
titanic_df_encoded = sex_encoder.fit(titanic_df_indexed).transform(titanic_df_indexed)

display(titanic_df_encoded)

In [0]:
# Cria coluna flag: 1.0 se era nulo, 0.0 se tinha valor
for col in colunas_numericas + colunas_categoricas:
    df = df.withColumn(
        f"{col}_flag_nulo",
        F.when(F.col(col).isNull(), 1.0).otherwise(0.0)
    )

colunas_flag = [f"{c}_flag_nulo" for c in colunas_numericas + colunas_categoricas]

In [0]:
imputer_num = Imputer(
    inputCols=colunas_numericas,
    outputCols=[f"{c}_imp" for c in colunas_numericas],
    strategy="mean"   # "mean" ou "median"
)

colunas_numericas_imp = [f"{c}_imp" for c in colunas_numericas]

In [0]:
# Passo 4a: StringIndexer para cada categórica
# handleInvalid="keep" para não quebrar nos nulos durante o fit
indexers = [
    StringIndexer(
        inputCol=col,
        outputCol=f"{col}_idx",
        handleInvalid="keep",    # trata nulos como categoria extra
        stringOrderType="frequencyDesc"
    )
    for col in colunas_categoricas
]

colunas_categoricas_idx = [f"{c}_idx" for c in colunas_categoricas]

# Passo 4b: Imputer pela moda nos índices
# O StringIndexer com handleInvalid="keep" atribui o último índice
# ao nulo. O Imputer com strategy="mode" vai substituir esse índice
# pelo índice mais frequente (moda real dos dados)
imputer_cat = Imputer(
    inputCols=colunas_categoricas_idx,
    outputCols=[f"{c}_imp" for c in colunas_categoricas_idx],
    strategy="mode"
)

colunas_categoricas_imp = [f"{c}_idx_imp" for c in colunas_categoricas]

# Passo 4c: OneHotEncoder nas categóricas imputadas
encoders = [
    OneHotEncoder(
        inputCol=f"{col}_idx_imp",
        outputCol=f"{col}_enc"
    )
    for col in colunas_categoricas
]

colunas_categoricas_enc = [f"{c}_enc" for c in colunas_categoricas]

In [0]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler, StringIndexer, OneHotEncoder, Imputer
from pyspark.ml.classification import LogisticRegression

# 1. Codificar coluna categórica
indexer = StringIndexer(inputCol="cidade", outputCol="cidade_idx")
encoder = OneHotEncoder(inputCol="cidade_idx", outputCol="cidade_enc")

# 2. Montar vetor
assembler = VectorAssembler(
    inputCols=["age", "income", "cidade_enc"],
    outputCol="features_raw"
)

# 3. Normalizar (opcional, mas recomendado)
scaler = StandardScaler(inputCol="features_raw", outputCol="features")

# 4. Modelo
lr = LogisticRegression(featuresCol="features", labelCol="churn")

# 5. Pipeline completo
pipeline = Pipeline(stages=[indexer, encoder, assembler, scaler, lr])
model = pipeline.fit(df_train)

In [0]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols = ['Age', 'Fare'],
    outputCol = 'features',
    handleInvalid='keep'
)

titanic_df_assembled = assembler.transform(titanic_df)

display(titanic_df_assembled)

In [0]:
assembler = VectorAssembler(
    inputCols=(
        colunas_numericas_imp      +   # numéricas imputadas pela média
        colunas_categoricas_enc    +   # categóricas imputadas pela moda + OHE
        colunas_flag                   # flags de nulidade (0 ou 1)
    ),
    outputCol="features",
    handleInvalid="error"   # aqui não deve sobrar nulos
)

In [0]:
pipeline = Pipeline(stages=[
    # Etapa 1: StringIndexer para cada categórica
    *indexers,

    # Etapa 2: Imputar numéricas pela média
    imputer_num,

    # Etapa 3: Imputar categóricas (índices) pela moda
    imputer_cat,

    # Etapa 4: OneHotEncoder nas categóricas imputadas
    *encoders,

    # Etapa 5: Montar vetor final
    assembler,
])

# Treina o pipeline (aprende médias, modas e vocabulários)
pipeline_model = pipeline.fit(df)

# Aplica as transformações
df_final = pipeline_model.transform(df)

# Visualiza o resultado
df_final.select("features").show(truncate=False)

# Modelagem